# 02 · Merge TUNIZI + TSAC → master dataset
**Owner:** Aziz · **Inputs:** `data/interim/tunizi_clean.csv` (Aziz), `data/interim/tsac_clean.csv` (Khaoula)
**Outputs:** `data/processed/master_clean.csv`, `logs/merge_log.csv`, `logs/script_check_30.csv`

Steps from the team spec: check columns → stack → cross-dataset duplicates → script detection (drop mixed) →
30-row hand check → derived columns → save → counts table for the group chat.

In [ ]:
# Works both locally (repo root or notebooks/) and in Colab after cloning the repo.
import os, sys
from pathlib import Path
if Path("notebooks").is_dir():
    ROOT = Path(".").resolve()
elif Path("../notebooks").is_dir():
    ROOT = Path("..").resolve()
else:  # Colab: clone first, e.g. !git clone https://github.com/<owner>/tunisian-sentiment.git
    ROOT = Path("/content/tunisian-sentiment")
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("Repo root:", ROOT)

In [ ]:
import os, re
import pandas as pd
from src.cleaning_utils import CleaningLog
from src.script_detect import detect_script

TUNIZI = ROOT / "data/interim/tunizi_clean.csv"
TSAC = Path(os.environ.get("TSAC_CLEAN", ROOT / "data/interim/tsac_clean.csv"))
OUT = ROOT / "data/processed/master_clean.csv"
LOG = ROOT / "logs/merge_log.csv"
CHECK = ROOT / "logs/script_check_30.csv"
SEED = 42
COLS = ["id", "text", "label", "source", "split"]
log = CleaningLog()
print("TSAC file:", TSAC)

## 1. Load and check both files have identical columns

In [ ]:
tz = pd.read_csv(TUNIZI, dtype=str, keep_default_na=False)
ts = pd.read_csv(TSAC, dtype=str, keep_default_na=False)
for name, d in [("tunizi", tz), ("tsac", ts)]:
    assert list(d.columns) == COLS, f"{name} columns are {list(d.columns)}, expected {COLS}"
    assert set(d.label) <= {"pos", "neg"}, f"{name}: unexpected labels {set(d.label)}"
    assert d.id.is_unique, f"{name}: duplicate ids"
    print(f"{name:7} {len(d):>6} rows", d.split.value_counts().to_dict())

## 2. Stack

In [ ]:
df = pd.concat([tz, ts], ignore_index=True)
log.add("stack tunizi_clean + tsac_clean", len(df), len(df), f"tunizi {len(tz)}, tsac {len(ts)}")

## 3. Cross-dataset duplicates
If a TUNIZI text also appears in TSAC, the **TUNIZI copy is dropped**. This protects the TSAC test set (no test comment can sit in training through TUNIZI) and keeps TSAC's script balance.

In [ ]:
tsac_labels = ts.drop_duplicates("text").set_index("text").label
dup = (df.source == "tunizi") & df.text.isin(tsac_labels.index)
dups = df[dup]
n_diff = (dups.label.values != tsac_labels.loc[dups.text].values).sum()
n_test = dups.text.isin(ts[ts.split == "test"].text).sum()
before = len(df); df = df[~dup]
log.add("drop TUNIZI copies of TSAC texts", before, len(df),
        f"{n_diff} had a different label; {n_test} matched a TSAC test comment")
dups[["id", "text", "label"]].head(10)

## 4. Script detection, drop mixed
Rule (`src/script_detect.py`): ≥ 80% Arabic letters → `arabic`, ≥ 80% Latin letters → `arabizi`, otherwise `mixed` (dropped and counted).

In [ ]:
df["script"] = df.text.map(detect_script)
print(pd.crosstab([df.source, df.split], df.script, margins=True))

tsac_share = df[df.source == "tsac"].script.value_counts(normalize=True).mul(100).round(1)
print("\nTSAC script split (%):", tsac_share.to_dict())
print("Proposal:               {'arabizi': 54, 'arabic': 44, 'mixed': 1}")

In [ ]:
mixed = df[df.script == "mixed"]
print("Examples of mixed rows:"); print(mixed.sample(min(8, len(mixed)), random_state=SEED)[["source", "text"]].to_string())
before = len(df); df = df[df.script != "mixed"]
log.add("drop mixed-script rows", before, len(df),
        "; ".join(f"{k} {v}" for k, v in mixed.source.value_counts().items()))

## 5. Hand check of the rule: 30 random rows (15 per script)
The rows are written to `logs/script_check_30.csv`. Open it, fill `correct` with `yes`/`no` for each row, then note the accuracy here and on the data slide.

In [ ]:
check = pd.concat([df[df.script == s].sample(15, random_state=SEED) for s in ("arabic", "arabizi")])
check = check[["id", "source", "script", "text"]].assign(correct="")
check.to_csv(CHECK, index=False, encoding="utf-8")
check[["id", "script", "text"]]

## 6. Derived columns
| Column | Definition |
|---|---|
| `n_chars` | number of characters |
| `n_words` | number of space-separated words |
| `has_emoji` | at least one emoji (Unicode emoji/pictograph ranges, regional flags, ❤ etc.) |
| `has_arabizi_digits` | a digit 2, 3, 5, 7, 8 or 9 touching a Latin letter (`3aychek` yes, `2016` no) |
| `has_elongation` | the same character 3+ times in a row (`ya3333jbni`) |

In [ ]:
EMOJI_RE = re.compile(
    "[\U0001F300-\U0001F5FF\U0001F600-\U0001F64F\U0001F680-\U0001F6FF\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF\U0001F800-\U0001F8FF\U0001F900-\U0001F9FF\U0001FA00-\U0001FAFF"
    "\U0001F1E6-\U0001F1FF☀-➿⬀-⯿⌀-⏿]"
)
ARABIZI_DIGIT_RE = re.compile(r"(?<=[A-Za-z])[235789]|[235789](?=[A-Za-z])")
ELONG_RE = re.compile(r"(.)\1{2,}")

df["n_chars"] = df.text.str.len()
df["n_words"] = df.text.str.split().str.len()
df["has_emoji"] = df.text.str.contains(EMOJI_RE)
df["has_arabizi_digits"] = df.text.str.contains(ARABIZI_DIGIT_RE)
df["has_elongation"] = df.text.map(lambda t: bool(ELONG_RE.search(t)))

# sanity examples from the spec
assert ARABIZI_DIGIT_RE.search("3aychek") and not ARABIZI_DIGIT_RE.search("2016")
assert ELONG_RE.search("ya3333jbni")
df.groupby("script")[["has_emoji", "has_arabizi_digits", "has_elongation"]].mean().round(3)

## 7. Save

In [ ]:
df = df[COLS + ["script", "n_chars", "n_words", "has_emoji", "has_arabizi_digits", "has_elongation"]]
assert df.id.is_unique and not df.text.eq("").any()
assert not df[df.split == "train"].text.isin(df[df.split == "test"].text).any(), "train/test overlap"
OUT.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT, index=False, encoding="utf-8")
log.add("final master_clean.csv", len(df), len(df))
log.save(LOG)
log.to_frame()

## 8. Counts for the group chat (source × split × script)

In [ ]:
counts = pd.crosstab([df.source, df.split], df.script, margins=True, margins_name="total")
print(counts.to_markdown() if hasattr(counts, "to_markdown") else counts)
counts